In [2]:
"""
diag_replay.py -- localise a golden-gate FAIL. Compares the batch feature path
against the worker Engine, channel by channel, for one session.

Order of blame:
  1. leg_dir / leg_start_jma      -- state parity (Stage-0 rules)
  2. z-channels                   -- numerical (_expanding_z vs Welford)
  3. events                       -- grammar parity (which class fired when)

Run diagnose_channels() first. If every channel matches to ~1e-12 the cause is
grammar, and diagnose_events() names it.
"""

import numpy as np
import pandas as pd

from worker_diag import (Engine, Welford, load_contract, MODEL_PATH, SC_TAG,
                    load_research_frame, ZWARM, TAUS)
from common import _expanding_z

# ---------------------------------------------------------------- CONFIG
BARS_PATH = f"stage-0/bars_{SC_TAG}.pqt"        # Stage-0 output for this tag
EVENTS_PATH = f"stage-0/events_{SC_TAG}.pqt"
# ----------------------------------------------------------------


def _session(src, bars, day):
    g = (src[src["timestamp"].dt.date == day]
         .sort_values("timestamp").reset_index(drop=True))
    b = (bars[bars["date"].astype(str) == str(day)]
         .sort_values("bar_index").reset_index(drop=True))
    assert len(g) == len(b), f"row mismatch src={len(g)} bars={len(b)}"
    assert np.array_equal(g["JMA"].to_numpy(), b["jma"].to_numpy()), \
        "JMA differs between source file and stage-0 bars"
    return g, b


def _batch_state(b):
    """Exactly augment_featurizer: leg_dir + leg_start_jma from stage-0 bars."""
    jma = b["jma"].to_numpy(np.float64)
    leg_dir = b["jma_leg_dir"].to_numpy(np.float64)
    tgt = b["is_target"].to_numpy()
    starts = np.where(tgt)[0]
    leg_id = np.cumsum(tgt)
    start_idx = np.concatenate(([0], starts))[leg_id]
    return leg_dir, jma, jma[start_idx]


def _worker_state(c, g):
    """Drive the Engine over the session, capturing its internal state per bar."""
    cols = ["rawOpen", "rawLast", "JMA"] + c["stream_cols"]
    V = g[cols].to_numpy(np.float64)
    eng = Engine(c)
    n = len(g)
    leg_dir = np.zeros(n)
    leg_start = np.zeros(n)
    fired = []                                   # (local_bar, class) worker recorded
    for i in range(n):
        before = {cl: eng.last_event[cl] for cl in c["classes"]}
        eng.ingest(i, *V[i])
        leg_dir[i] = eng.jma_track.sign
        leg_start[i] = eng.leg_start_jma
        for cl in c["classes"]:
            if eng.last_event[cl] != before[cl]:
                fired.append((i, cl))
    return leg_dir, leg_start, fired


def diagnose_channels(day, src=None, bars=None):
    c = load_contract(MODEL_PATH)
    src = load_research_frame(c) if src is None else src
    bars = pd.read_parquet(BARS_PATH) if bars is None else bars
    g, b = _session(src, bars, day)

    bd, jma, bstart = _batch_state(b)
    wd, wstart, _ = _worker_state(c, g)

    print(f"==== {day}  ({len(g)} bars) ====")
    nd = int((bd != wd).sum())
    ns = int((np.abs(bstart - wstart) > 0).sum())
    print(f"leg_dir       mismatch bars: {nd}"
          + (f"   first at {int(np.argmax(bd != wd))}" if nd else ""))
    print(f"leg_start_jma mismatch bars: {ns}"
          + (f"   first at {int(np.argmax(np.abs(bstart - wstart) > 0))}" if ns else ""))
    if nd or ns:
        print("STATE PARITY BROKEN -- fix before reading channels below")

    osc = [g[col].to_numpy(np.float64) for col in c["stream_cols"]]
    body = g["rawLast"].to_numpy(np.float64) - g["rawOpen"].to_numpy(np.float64)

    chans = {}
    for i, col in enumerate(c["stream_cols"]):
        chans[f"z_{col}_signed"] = osc[i] * bd
    for i, col in enumerate(c["stream_cols"]):
        chans[f"z_{col}_mag"] = np.abs(osc[i])
    chans["z_leg_amp"] = np.abs(jma - bstart)
    chans["z_body_raw_signed"] = body * bd
    chans["z_body_raw_mag"] = np.abs(body)

    print(f"{'channel':24s} {'max|d|':>10s} {'n>1e-9':>8s} {'n>1e-6':>8s} {'worst':>7s}")
    rows = []
    for name, x in chans.items():
        zb = _expanding_z(x, ZWARM)
        w = Welford()
        zw = np.array([w.z_then_update(v) for v in x])
        D = np.abs(zb - zw)
        rows.append((name, float(D.max()), int((D > 1e-9).sum()),
                     int((D > 1e-6).sum()), int(np.argmax(D))))
        print(f"{name:24s} {D.max():10.3e} {int((D > 1e-9).sum()):8d} "
              f"{int((D > 1e-6).sum()):8d} {int(np.argmax(D)):7d}")
    return pd.DataFrame(rows, columns=["channel", "max_abs_d", "n_gt_1e9",
                                       "n_gt_1e6", "worst_bar"])


def diagnose_events(day, src=None, bars=None, events=None):
    """Compare which (class, local_bar) the worker records vs stage-0 events."""
    c = load_contract(MODEL_PATH)
    src = load_research_frame(c) if src is None else src
    bars = pd.read_parquet(BARS_PATH) if bars is None else bars
    events = pd.read_parquet(EVENTS_PATH) if events is None else events
    g, b = _session(src, bars, day)
    _, _, fired = _worker_state(c, g)

    first = int(b["bar_index"].iloc[0])
    e = events[events["date"].astype(str) == str(day)].copy()
    e["local"] = e["event_bar"].to_numpy() - first

    batch = set()
    for _, r in e.iterrows():
        if r["stream"] == c["self"]:
            batch.add((int(r["local"]), (c["self"], "all")))
        elif r["opposing"] != 0:
            batch.add((int(r["local"]),
                       (r["stream"], "opp" if r["opposing"] == 1 else "conf")))
    work = set(fired)

    only_w = sorted(work - batch)
    only_b = sorted(batch - work)
    print(f"==== {day} events ====")
    print(f"batch {len(batch)}   worker {len(work)}   "
          f"worker-only {len(only_w)}   batch-only {len(only_b)}")
    for tag, lst in (("worker-only", only_w), ("batch-only", only_b)):
        for bar, cl in lst[:15]:
            print(f"  {tag:12s} local_bar={bar:5d}  {cl}")
    return only_w, only_b


def _batch_X(c, g, b, e):
    """Rebuild batch X for one session, self-contained from stage-0 outputs.
    Mirrors stage_5.build_grammar_features + build_value_features exactly."""
    from scipy.signal import lfilter
    n = len(g)
    first = int(b["bar_index"].iloc[0])
    tgt = b["is_target"].to_numpy()
    warm = b["warm"].to_numpy()
    t = np.nonzero(~warm)[0]

    ts = pd.DatetimeIndex(b["timestamp"])
    s0 = pd.Timestamp(c["session_start"])
    smin = s0.hour * 60 + s0.minute
    mins = ts.hour.to_numpy() * 60 + ts.minute.to_numpy() - smin
    tod = np.clip(mins // TOD_BIN_MIN, 0, c["n_tod"] - 1).astype(np.int16)
    lt_incl = np.maximum.accumulate(np.where(tgt, np.arange(n), -1))

    ee = e[e["date"].astype(str) == str(b["date"].iloc[0])]
    cols = []
    for (s, kind) in c["classes"]:
        if s == c["self"]:
            sub = ee[ee["stream"] == s]
        else:
            want = 1 if kind == "opp" else -1
            sub = ee[(ee["stream"] == s) & (ee["opposing"] == want)]
        ind = np.zeros(n, np.float64)
        ind[sub["event_bar"].to_numpy() - first] = 1.0
        occ = np.where(ind > 0, np.arange(n), -1)
        last = np.maximum.accumulate(occ)
        lastm1 = np.concatenate(([-1], last[:-1]))
        bsince = np.where(lastm1 >= 0, np.arange(n) - lastm1, np.arange(n) + 1)
        cols.append(bsince[t])
        x = np.concatenate(([0.0], ind[:-1]))
        for tau in TAUS:
            a = np.exp(-1.0 / tau)
            cols.append(lfilter([a], [1.0, -a], x)[t])
    lt = np.where(t > 0, lt_incl[np.maximum(t - 1, 0)], -1)
    cols.append(np.where(lt >= 0, t - lt, t + 1))
    cols.append(tod[t].astype(np.float64))
    Xg = np.stack(cols, 1).astype(np.float32)

    leg_dir, jma, leg_start = _batch_state(b)
    osc = [g[col].to_numpy(np.float64) for col in c["stream_cols"]]
    body = g["rawLast"].to_numpy(np.float64) - g["rawOpen"].to_numpy(np.float64)
    feats = [_expanding_z(o * leg_dir, ZWARM) for o in osc]
    feats += [_expanding_z(np.abs(o), ZWARM) for o in osc]
    feats.append(_expanding_z(np.abs(jma - leg_start), ZWARM))
    feats.append(_expanding_z(body * leg_dir, ZWARM))
    feats.append(_expanding_z(np.abs(body), ZWARM))
    M = np.stack(feats, 1)
    M = np.concatenate([np.zeros((1, M.shape[1])), M[:-1]], 0)
    lagged = [M]
    for L in VALUE_LAGS:
        lagged.append(np.concatenate([np.zeros((L, M.shape[1])), M[:-L]], 0))
    M = np.concatenate(lagged, 1)
    Mt = M[t]
    Mt[(t < ZWARM + max(VALUE_LAGS))] = 0.0
    return np.hstack([Xg, Mt.astype(np.float32)]), t


def _worker_X(c, g):
    cols = ["rawOpen", "rawLast", "JMA"] + c["stream_cols"]
    V = g[cols].to_numpy(np.float64)
    eng = Engine(c)
    rows, ts = [], []
    for i in range(len(g)):
        _, f = eng.ingest(i, *V[i])
        if f is not None:
            rows.append(f)
            ts.append(i + 1)
    return np.vstack(rows).astype(np.float32), np.array(ts)


def diagnose_features(day, bad_k=None, src=None, bars=None, events=None):
    """Diff batch X vs worker X for one session. bad_k = list of local forming-bar
    indices (t) known to mispredict; if given, the per-feature report is for those
    rows only."""
    c = load_contract(MODEL_PATH)
    src = load_research_frame(c) if src is None else src
    bars = pd.read_parquet(BARS_PATH) if bars is None else bars
    events = pd.read_parquet(EVENTS_PATH) if events is None else events
    g, b = _session(src, bars, day)

    Xb, tb = _batch_X(c, g, b, events)
    Xw, tw = _worker_X(c, g)
    assert np.array_equal(tb, tw), f"row grid differs: {tb[:5]} vs {tw[:5]}"

    names = worker_feature_names(c)
    D = np.abs(Xb.astype(np.float64) - Xw.astype(np.float64))
    print(f"==== {day}  X {Xb.shape}  (float32 both sides) ====")
    print(f"rows with ANY float32 difference: {int((D > 0).any(axis=1).sum())} / {len(D)}")
    print(f"{'feature':32s} {'max|d|':>10s} {'n!=0':>8s}")
    order = np.argsort(-D.max(axis=0))
    for i in order[:12]:
        nz = int((D[:, i] > 0).sum())
        if D[:, i].max() == 0:
            break
        print(f"{names[i]:32s} {D[:, i].max():10.3e} {nz:8d}")

    if bad_k is not None and len(bad_k):
        sel = np.isin(tb, np.asarray(bad_k))
        print(f"---- failing rows only ({int(sel.sum())}) ----")
        Db = D[sel]
        for i in np.argsort(-Db.max(axis=0))[:8]:
            if Db[:, i].max() == 0:
                break
            print(f"{names[i]:32s} {Db[:, i].max():10.3e} "
                  f"{int((Db[:, i] > 0).sum()):8d}")
    return Xb, Xw, tb, names

In [3]:
# ---------------------------------------------------------------- usage

print(sorted(bad.date.unique())[:10])
DAY = bad.date.iloc[0]
ch = diagnose_channels(DAY, src=src)
ow, ob = diagnose_events(DAY, src=src)
bk = bad[bad.date == DAY].k.to_numpy() + 10        # k is 0-based on scored rows
Xb, Xw, tb, names = diagnose_features(DAY, bad_k=bk, src=src)

NameError: name 'bad' is not defined

In [ ]:
'2022-01-17', '2022-07-04', '2022-11-24', '2023-01-16',
               '2023-02-20', '2023-04-07', '2023-05-29', '2023-06-19',
               '2023-07-04', '2023-09-04', '2023-11-23', '2024-02-19',
               '2024-05-27', '2024-06-19', '2024-07-04', '2024-11-28',
               '2025-01-09', '2025-07-04', '2025-09-01', '2025-11-27',
               '2026-04-03', '2026-07-09'],